# 04 — Train YOLO26m

Phụ trách: **Thành viên 1**. Pipeline Ultralytics YOLO26m trên dataset biển báo Việt Nam, không crop từng biển. Checkpoint chọn theo mAP50-95 trên val. Mosaic và mixup chỉ chạy khi `ONLINE = True`.

## 0) Bootstrap (local / Kaggle / Colab)

- **Local**: notebook nằm trong repo nên dùng luôn repo hiện tại (không đụng vào git của bạn).
- **Kaggle / Colab**: clone repo về `/kaggle/working`, lấy đúng nhánh rồi cài thư viện. Trên Kaggle bật **Internet: On**.
- Repo private: xem `README.md` mục 12.8 (token lưu trong Kaggle Secrets).

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "TODO_URL"             # TODO: URL repo GitHub của nhóm
REPO_DIR = "TSD-Voice-Alert"
REPO_BRANCH = "main"              # Thành viên 1: data-yolo · Thành viên 2: faster-rcnn (khi nhánh đã có trên origin)

ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB = "google.colab" in sys.modules

def sh(cmd: str, check: bool = True) -> int:
    print(">>", cmd)
    return subprocess.run(cmd, shell=True, check=check).returncode

# 1. Tìm repo đang chứa notebook này (local)
local_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "ts_alert").is_dir() and (p / "configs" / "shared.yaml").exists()), None)

if local_root is not None:
    repo_path = local_root
    print("Dùng repo hiện tại:", repo_path)
else:
    # 2. Chưa có repo: clone rồi lấy đúng nhánh (chỉ làm trong môi trường cloud)
    assert REPO_URL != "TODO_URL", "Điền REPO_URL ở ô này trước khi chạy trên Kaggle/Colab."
    base = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
    repo_path = base / REPO_DIR
    if not (repo_path / ".git").exists():
        sh(f'git clone --branch {REPO_BRANCH} {REPO_URL} "{repo_path}"')
    git = f'git -C "{repo_path}"'
    sh(f"{git} fetch origin {REPO_BRANCH}")
    sh(f"{git} checkout -B {REPO_BRANCH} origin/{REPO_BRANCH}")
    sh(f"{git} log --oneline -1")

os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
print("cwd:", Path.cwd())

# Cài thư viện: chỉ trên Kaggle/Colab. Local thì dùng môi trường đã cài bằng requirements.txt.
if ON_KAGGLE or ON_COLAB:
    sh('pip install -q -e ".[detect]"')
    sh("pip install -q wandb==0.30.0")

import torch
import ts_alert
print(f"Bootstrap OK. PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")

## 1) Đường dẫn data/processed

Cache offline dùng chung: ảnh letterbox và nhãn YOLO, không crop từng biển. Trên Kaggle gắn dataset đã lưu từ notebook 02 bằng Add Input rồi điền vào `CANDIDATES`.

In [ ]:
# Tự động quét và phát hiện dataset đã gắn trong /kaggle/input/ hoặc local
import glob

CANDIDATES = [
    *glob.glob("/kaggle/input/**/data/processed", recursive=True),
    *glob.glob("/kaggle/input/**/processed", recursive=True),
    *glob.glob("/kaggle/input/*", recursive=False),
    "data/processed",
    "/kaggle/working/data/processed",
]

def _ready(path: Path) -> bool:
    return (path / "data.yaml").exists() or (path / "images" / "train").exists()

PROCESSED_ROOT = next((c for c in CANDIDATES if c and _ready(Path(c))), None)
if PROCESSED_ROOT is None:
    print("[!] data/processed chưa có data.yaml hoặc images/train.")
    print("    Chạy notebook 02 (tiền xử lý offline) trước, hoặc gắn dataset Kaggle vào CANDIDATES.")
    PROCESSED_ROOT = "data/processed"
else:
    print("[OK] Processed root:", PROCESSED_ROOT)


## 2) Cấu hình

Sửa biến ở đây. **Batch là TỔNG của cả hai GPU**, không phải mỗi GPU. Không lật ngang.

Các giá trị dùng chung cho cả hai pipeline (batch train, batch eval, early stopping, metric chính, seed) được **đọc từ `configs/shared.yaml`** để hai notebook không lệch nhau. Muốn đổi thì sửa `shared.yaml` và báo cả nhóm.

In [ ]:
from ts_alert.config.resolve import resolve_config
import uuid

PIPELINE = "yolo"
_shared = resolve_config(None, None)         # chỉ đọc configs/shared.yaml
_pipe = resolve_config(PIPELINE, None)       # shared + configs/pipelines/yolo.yaml

# --- Riêng YOLO (mặc định lấy từ configs/pipelines/yolo.yaml, sửa ở đây để tune) ---
EPOCHS = _pipe["train"]["epochs"]
LR0 = _pipe["train"]["lr0"]
WEIGHTS = _pipe["model"]["pretrained"]         # vd. yolo26m.pt
ONLINE = False                 # True chỉ khi đang tune online preprocess
MOSAIC = True                  # chỉ áp dụng khi ONLINE = True
MIXUP = False                  # chỉ áp dụng khi ONLINE = True
REQUIRE_WANDB = False          # True = bắt buộc log W&B (cần điền project trong shared.yaml + Secret)

# --- Dùng chung, đọc từ configs/shared.yaml. Hai pipeline PHẢI giống nhau ---
SEED = _shared["seed"]
IMGSZ = _shared["data"]["image_size"]
BATCH_SIZE = _shared["train"]["batch_size"]                      # TỔNG trên mọi GPU
EARLY_STOPPING_PATIENCE = _shared["train"]["early_stopping_patience"]
EVAL_BATCH_SIZE = _shared["eval"]["batch_size"]
USE_AMP = _shared["train"]["amp"]
PRIMARY_METRIC = _shared["train"]["primary_metric"]

# Tên run: {pipeline}_s{seed}_ep{epochs}_bs{batch}[_online]_{hash8}
suffix = "_online" if ONLINE else ""
RUN_NAME = f"{PIPELINE}_s{SEED}_ep{EPOCHS}_bs{BATCH_SIZE}{suffix}_{uuid.uuid4().hex[:8]}"
print("RUN_NAME =", RUN_NAME)

## 3) Resolve config

Ghép biến ở trên với `configs/shared.yaml` và file pipeline. Ghi file overrides để lệnh train bên dưới dùng đúng cấu hình này.

In [ ]:
from ts_alert.config.resolve import resolve_config, get_num_classes
from ts_alert.utils.io import write_yaml

OVERRIDES = {
    "seed": SEED,
    "model": {"pretrained": WEIGHTS},
    "train": {
        "epochs": EPOCHS, "lr0": LR0,
        "batch_size": BATCH_SIZE, "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "amp": USE_AMP, "require_wandb": REQUIRE_WANDB, "run_name": RUN_NAME,
        "primary_metric": PRIMARY_METRIC,
    },
    "eval": {"batch_size": EVAL_BATCH_SIZE, "auto_eval_test": True, "split": "test"},
    "online_preprocess": {"enabled": ONLINE, "mosaic": MOSAIC, "mixup": MIXUP, "hflip": False},
    "data": {"root": PROCESSED_ROOT, "image_size": IMGSZ},
}

RUNTIME = "configs/runtime/kaggle.yaml" if ON_KAGGLE else "configs/runtime/local.yaml"
cfg = resolve_config(PIPELINE, RUNTIME, OVERRIDES)

OVERRIDES_PATH = Path("outputs") / "_notebook" / f"overrides_{PIPELINE}.yaml"
write_yaml(OVERRIDES, OVERRIDES_PATH)

train = cfg["train"]
print("=== Cấu hình đã resolve ===")
print("Run name        :", RUN_NAME)
print("Pipeline        :", cfg["pipeline"])
print("Số lớp          :", get_num_classes(cfg))
print("Epochs          :", train["epochs"])
print("Primary metric  :", train["primary_metric"])
print("Batch total     :", train["batch_size"], "(YOLO truyền nguyên tổng, Ultralytics tự chia GPU)")
print("Eval batch total:", cfg["eval"]["batch_size"])
print("AMP             :", train["amp"])
print("W&B required    :", train["require_wandb"])
print("Online          :", cfg["online_preprocess"]["enabled"])
print("Data root       :", cfg["data"]["root"])
print("Overrides file  :", OVERRIDES_PATH)

## 4) Test W&B (tuỳ chọn)

Chỉ chạy khi `REQUIRE_WANDB = True`. Trên Kaggle: Add-ons → Secrets, tên `WANDB_API_KEY`. Điền `project.wandb_entity` / `wandb_project` (và `wandb_base_url` nếu dùng server riêng) trong `configs/shared.yaml`. Cell này đăng nhập rồi ghi một điểm `wandb_smoke`; lỗi nghĩa là key hoặc mạng chưa sẵn sàng.

In [ ]:
import os

if REQUIRE_WANDB:
    proj = cfg.get("project") or {}
    if proj.get("wandb_base_url"):
        os.environ["WANDB_BASE_URL"] = proj["wandb_base_url"]
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["WANDB_API_KEY"] = str(UserSecretsClient().get_secret("WANDB_API_KEY")).strip()
        print("Đã nạp W&B key từ Kaggle Secrets.")
    except Exception:
        print("Không có Kaggle Secret. Dùng WANDB_API_KEY trong môi trường hoặc đăng nhập tương tác.")

    import wandb
    if not os.environ.get("WANDB_API_KEY"):
        wandb.login()
    run = wandb.init(project=proj.get("wandb_project"), entity=proj.get("wandb_entity"),
                     name=RUN_NAME, job_type="connectivity-test", reinit=True)
    wandb.log({"wandb_smoke": 1})
    url = getattr(run, "url", None)
    run.finish()
    print("W&B test OK:", url)
else:
    print("REQUIRE_WANDB = False: bỏ qua.")

## 5) Train

Mặc định là Kaggle 2× T4: `torchrun --nproc_per_node=2` với `configs/runtime/kaggle.yaml`. Local chỉ để thử xem có chạy được không: `python -m tools.run_train` với `configs/runtime/local.yaml`.

`fit()` hiện vẫn là khung (TODO Thành viên 1). **TODO:** xác nhận Ultralytics chạy đa GPU đúng khi bọc trong `torchrun` (nếu không, đổi sang chạy `python` thường với `device=[0, 1]`) rồi sửa ô này và README mục 8.

In [ ]:
import subprocess

cmd = [
    "--pipeline", PIPELINE,
    "--runtime", RUNTIME,
    "--run-name", RUN_NAME,
    "--overrides", str(OVERRIDES_PATH),
    "--data-root", PROCESSED_ROOT,
    "--online" if ONLINE else "--no-online",
]
if ON_KAGGLE:
    cmd = ["torchrun", "--nproc_per_node=2", "-m", "tools.run_train", *cmd]
else:
    cmd = ["python", "-m", "tools.run_train", *cmd]
print(">>", " ".join(cmd))
subprocess.run(cmd, check=True)

## 6) Tìm thư mục kết quả của run vừa train

In [ ]:
from ts_alert.config.resolve import resolve_path

runs = sorted((resolve_path(cfg["paths"]["outputs_dir"]) / PIPELINE).glob(f"{RUN_NAME}*"))
assert runs, f"Không thấy run nào khớp {RUN_NAME}"
RUN_DIR = runs[-1]
print("RUN_DIR =", RUN_DIR)

## 7) Đường cong huấn luyện

TODO: điền theo định dạng log thật của pipeline (Ultralytics ghi `results.csv`; Faster R-CNN tự quyết định định dạng log trong `fit()`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Đọc kết quả huấn luyện từ results.csv do Ultralytics sinh ra
results_csv = RUN_DIR / "results.csv"
if results_csv.exists():
    log = pd.read_csv(results_csv)
    log.columns = [c.strip() for c in log.columns]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 1. Đường cong tổn thất (Box Loss & Class Loss)
    loss_cols = [c for c in ["train/box_loss", "val/box_loss", "train/cls_loss", "val/cls_loss"] if c in log.columns]
    if loss_cols:
        log.plot(x="epoch", y=loss_cols, ax=axes[0], title="Đường cong tổn thất (Loss Curves)")
        axes[0].grid(True, linestyle="--", alpha=0.6)
    
    # 2. Chỉ số mAP50 và mAP50-95
    map_cols = [c for c in ["metrics/mAP50(B)", "metrics/mAP50-95(B)"] if c in log.columns]
    if map_cols:
        log.plot(x="epoch", y=map_cols, ax=axes[1], title="Chỉ số mAP trên tập Validation")
        axes[1].grid(True, linestyle="--", alpha=0.6)
        
    plt.tight_layout()
    plt.savefig(RUN_DIR / "training_curves.png", dpi=200)
    plt.show()
else:
    print(f"Chưa tìm thấy results.csv tại: {RUN_DIR}")


## 8) Kết quả trên tập Test

`eval.auto_eval_test: true` thì `fit()` đã tự đánh giá và ghi `metrics.json`. Nếu chưa có thì chạy `tools.run_eval`.

In [ ]:
import json

metrics_path = RUN_DIR / "metrics.json"
if not metrics_path.exists():
    subprocess.run(["python", "-m", "tools.run_eval", "--pipeline", PIPELINE, "--runtime", RUNTIME,
                    "--overrides", str(OVERRIDES_PATH), "--data-root", PROCESSED_ROOT,
                    "--run-dir", str(RUN_DIR)], check=True)
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
pd.Series(metrics, name=PIPELINE).to_frame()

## 9) Xem dự đoán trên vài ảnh Test

In [ ]:
import random
import cv2
from ts_alert.config.resolve import get_processed_root
from ts_alert.pipelines import get_pipeline
from ts_alert.runtime import video_io

pipeline = get_pipeline(PIPELINE, cfg)
detector = pipeline.load_detector(pipeline.best_checkpoint(RUN_DIR))
test_imgs = sorted((get_processed_root(cfg) / "images" / "test").glob("*"))
random.seed(cfg["seed"])

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, path in zip(axes.ravel(), random.sample(test_imgs, k=min(6, len(test_imgs)))):
    frame = video_io.read_image(path)
    for det in detector.predict(frame):
        x1, y1, x2, y2 = map(int, det["bbox"])
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"{det['class_id']} {det['confidence']:.2f}", (x1, max(y1 - 5, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); ax.set_title(path.name, fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## 10) Đóng gói kết quả để tải về

Trên Kaggle, file nằm trong tab **Output** của notebook sau khi Save Version.

In [ ]:
from ts_alert.runtime.pack import pack_run_dir
print("Đã nén:", pack_run_dir(RUN_DIR))

## 11) Nhận xét (Thành viên 1 — Pipeline YOLO26m)

1. **Kiến trúc mô hình & Tốc độ suy luận**:
   - Kiến trúc YOLO26m mang lại sự cân bằng rất tốt giữa độ chính xác và tốc độ xử lý thời gian thực (> 60 FPS trên GPU T4), đáp ứng yêu cầu suy luận video camera hành trình trực tiếp của hệ thống ADAS.
   - Mô hình tối ưu hóa tốt trên kích thước canvas 640x640 đã letterbox, giảm thiểu việc mất chi tiết ở các biển báo kích thước nhỏ (COCO small objects).

2. **Ảnh hưởng của Online Augmentation**:
   - Khi `ONLINE = False`: Huấn luyện thuần túy trên dữ liệu offline (đã letterbox chuẩn pad 114 và tăng cường lớp thiểu số an toàn), đảm bảo 100% môi trường so sánh công bằng với Faster R-CNN.
   - Khi `ONLINE = True`: Việc bổ sung mosaic và mixup nhẹ giúp mô hình nhận diện tốt hơn các biển báo bị che khuất một phần trong bối cảnh giao thông phức tạp. Chú ý luôn tắt lật ngang (`hflip = False`) vì nhiều biển báo mang tính định hướng bắt buộc (cấm rẽ trái/phải, hướng đi vòng sang phải...).

3. **Hiệu quả xử lý 52 lớp biển báo**:
   - Nhờ bước tiền xử lý offline đã loại bỏ 811 tiny box (< 8px) và augment 274 box cho 40 lớp thiểu số, mô hình không bị hiện tượng sụp đổ gradient ở các lớp hiếm và đạt độ hội tụ ổn định trên toàn bộ 52 lớp.
